In [11]:
# ============================================================
# 🎙️ Llama 3 Professional Analyst Station — Kaggle T4
# Исправленная и оптимизированная версия
# ============================================================

# --- УСТАНОВКА ЗАВИСИМОСТЕЙ ---
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
!pip install -q deep-translator gradio
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q fpdf2 python-docx markdown
!apt-get install -y fonts-dejavu-core

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-dejavu-core is already the newest version (2.37-2build1).
0 upgraded, 0 newly installed, 0 to remove and 134 not upgraded.


In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [13]:
# !pip install -q "pyannote.audio==2.1.1"
# !pip install -q "torchaudio==2.0.2" --force-reinstall

In [14]:
# !pip install -q "numpy==2.0.2" --force-reinstall
# !pip install -q "numba==0.60.0" --force-reinstall

In [15]:
# --- ИМПОРТЫ ---
import os
import time
import torch
import whisper
import gradio as gr

from threading import Thread
from concurrent.futures import ThreadPoolExecutor

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)
from deep_translator import GoogleTranslator
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
# from pyannote.audio import Pipeline
# print("✅ pyannote загружен успешно")

In [16]:
# ============================================================
# --- CONSTANTS ---
# ============================================================
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
WHISPER_MODEL_SIZE = "small"   # base / small / medium — on T4 "small" is optimal
MAX_NEW_TOKENS_GRAMMAR = 2500
MAX_NEW_TOKENS_ANALYSIS = 1200
TRANSLATION_INTERVAL_SEC = 1.5


In [17]:
ROLES = {
    "📊 Analyst":    "You are a professional analyst. Extract core essence and key points. Use Markdown.",
    "⚔️ Critic":      "You are a critical opponent. Find weak arguments, logical flaws and inconsistencies.",
    "📱 SMM manager":"You are an SMM manager. Create an engaging Telegram post based on this content.",
    "📋 Secretary":   "You are a secretary. Create a meeting protocol: who said what, what tasks were assigned.",
    "🧠 Psychologist":    "You are a psychologist. Analyze the emotional tone, stress points and speaker's mental state.",
    "🎯 Coach":        "You are a life coach. Extract actionable insights and next steps from this content.",
}

In [18]:
# ============================================================
# --- AUTHORIZATION HuggingFace ---
# ============================================================
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("✅ HuggingFace authorization successful")
else:
    raise RuntimeError("❌ Secret HF_TOKEN not found. Add it to -> Secrets")

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
✅ HuggingFace авторизация успешна


In [19]:
# ============================================================
# --- AUDIO FILE CHECK (optional) ---
# ============================================================
audio_filename = "/kaggle/input/audiotest/denver_extract.mp3"

if os.path.exists(audio_filename):
    print(f"✅ File {audio_filename} found")
else:
    print(f"⚠️  File not found: {audio_filename}")
    available = os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else []
    print("Available datasets:", available)


⚠️  Файл не найден: /kaggle/input/audiotest/denver_extract.mp3
Доступные датасеты: []


In [20]:

# ============================================================
# --- LOADING MODELS ---
# ============================================================
print(f"🔄 Loading Whisper '{WHISPER_MODEL_SIZE}'...")
stt_model = whisper.load_model(WHISPER_MODEL_SIZE)
print("✅ Whisper loaded")


# print("🔄 Загружаем Speaker Diarization...")
# diarization_pipeline = Pipeline.from_pretrained(
#     "pyannote/speaker-diarization-3.1",
#     use_auth_token=hf_token
# )
# diarization_pipeline.to(torch.device("cuda"))
# print("✅ Speaker Diarization загружен")

🔄 Загружаем Whisper 'small'...


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 209MiB/s]


✅ Whisper загружен


In [21]:
# 4-bit quantization for savings VRAM on T4 (16 GB)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


In [22]:
print(f"🔄 Загружаем {LLAMA}...")
tokenizer = AutoTokenizer.from_pretrained(LLAMA, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()  # Disabling dropout - speeds up inference
print("✅ LLaMA загружена")

🔄 Загружаем meta-llama/Llama-3.2-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ LLaMA загружена


In [ ]:
# Executor for asynchronous translation (does not block streaming)
_translator_executor = ThreadPoolExecutor(max_workers=1)

# ============================================================
# --- AUXILIARY FUNCTIONS ---
# ============================================================

def _translate_async(text: str) -> str:
    try:
        return GoogleTranslator(source="en", target="ru").translate(text)
    except Exception:
        return text

def _clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# --- PROCESSING LOGIC ---
# ============================================================

def transcribe_raw(audio_path: str) -> str:
    """The standard transcript is raw text without timestamps for LLaMA."""
    if audio_path is None:
        return ""
    result = stt_model.transcribe(audio_path, fp16=True)
    return result["text"]

def transcribe_with_timestamps(audio_path: str) -> str:
    """Transcript with timecodes.Transcript with timecodes."""
    if audio_path is None:
        return ""
    result = stt_model.transcribe(audio_path, fp16=True, word_timestamps=True)
    output = ""
    for segment in result["segments"]:
        start = int(segment["start"])
        mins, secs = divmod(start, 60)
        output += f"[{mins:02d}:{secs:02d}] {segment['text'].strip()}\n"
    return output

def transcribe_auto(audio_path: str, mode: str) -> str:
    """Router - selects the mode using the switch."""
    if audio_path is None:
        return ""
    if mode == "👥 Identify speakers":
        return "⚠️ Speaker Diarization is temporarily unavailable."
    elif mode == "🕐 With timecodes":
        return transcribe_with_timestamps(audio_path)
    else:
        return transcribe_raw(audio_path)

# def transcribe_with_speakers(audio_path: str) -> str:
#     """Transcript with speaker identification via Pyannote."""
#     if audio_path is None:
#         return ""

# def transcribe_auto(audio_path: str, mode: str) -> str:
#     """Selects the decryption mode by the switch."""
#     if mode == "👥 Identify speakers":
#         return transcribe_with_speakers(audio_path)
#     return transcribe_raw(audio_path)


    
#     # Step 1: Determine who speaks and when
#     diarization = diarization_pipeline(audio_path)

#     # Step 2: Full transcript via Whisper with timecodes
#     whisper_result = stt_model.transcribe(
#         audio_path,
#         fp16=True,
#         word_timestamps=True,
#     )

#     # Step 3: Align Whisper segments with speaker labels
#     output_lines = []

#     for segment in whisper_result["segments"]:
#         seg_start = segment["start"]
#         seg_end   = segment["end"]
#         seg_text  = segment["text"].strip()

#         # We find the speaker who spoke for most of this segment.
#         speaker = "Speaker?"
#         max_overlap = 0.0

#         for turn, _, spk in diarization.itertracks(yield_label=True):
#             overlap = min(turn.end, seg_end) - max(turn.start, seg_start)
#             if overlap > max_overlap:
#                 max_overlap = overlap
#                 # Format: SPEAKER_00 → Speaker 1
#                 num = int(spk.split("_")[-1]) + 1
#                 speaker = f"Speaker {num}"

#         # Formatting timecode [MM:SS]
#         mins = int(seg_start) // 60
#         secs = int(seg_start) % 60
#         timestamp = f"[{mins:02d}:{secs:02d}]"

#         output_lines.append(f"**{speaker}** {timestamp}: {seg_text}")

#     return "\n\n".join(output_lines)


def fix_grammar_llama(raw_text: str) -> str:
    if not raw_text.strip():
        return "### ⚠️ There is no text to process."

    prompt = (
        "Fix grammar and punctuation only. "
        "Keep all details and original meaning exactly. "
        "Format the text into logical paragraphs with line breaks between speakers and topics. "
        "Return only the corrected text without any commentary:\n\n"
        f"{raw_text}"
    )
    messages = [{"role": "user", "content": prompt}]

    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    input_ids = encoded.to("cuda")
    attention_mask = torch.ones_like(input_ids).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS_GRAMMAR,
            use_cache=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    result = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    _clear_cuda_cache()
    return result


def generate_analysis_stream(user_text: str, custom_query: str, target_lang: str, role: str):
    if not user_text.strip():
        yield "### ⚠️ There is no text. Please transcribe the audio first."
        return

    # We take the system prompt from the dictionary for the selected role
    system_instr = ROLES.get(role, ROLES["📊 Analyst"])

    if custom_query.strip():
        # If there is a question, the role is ignored and we answer the question.
        system_instr = (
            "You are a precise assistant. "
            "Answer the user's SPECIFIC QUESTION based on the provided transcript. "
            "Give a direct, concise answer only."
        )
        user_task = f"Answer this question: '{custom_query}'\n\nTRANSCRIPT:\n{user_text}"
    else:
        user_task = f"Provide your analysis.\n\nTRANSCRIPT:\n{user_text}"

    messages = [
        {"role": "system", "content": system_instr},
        {"role": "user",   "content": user_task},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    input_ids = encoded.to("cuda")
    attention_mask = torch.ones_like(input_ids).to("cuda")

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    gen_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "streamer": streamer,
        "max_new_tokens": MAX_NEW_TOKENS_ANALYSIS,
        "temperature": 0.4,
        "do_sample": True,
        "use_cache": True,
        "repetition_penalty": 1.1,
        "pad_token_id": tokenizer.eos_token_id,
    }

    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    full_en = ""
    tr_cache = ""
    last_tr_time = time.time()
    tr_future = None

    for new_chunk in streamer:
        full_en += new_chunk

        if target_lang == "en":
            yield full_en + " ▌"
            continue

        now = time.time()
        sentence_end = any(c in new_chunk for c in ".!?\n")
        should_translate = (now - last_tr_time > TRANSLATION_INTERVAL_SEC) or sentence_end

        if should_translate and (tr_future is None or tr_future.done()):
            tr_future = _translator_executor.submit(_translate_async, full_en)
            last_tr_time = now

        if tr_future is not None and tr_future.done():
            try:
                tr_cache = tr_future.result()
            except Exception:
                pass

        yield (tr_cache if tr_cache else full_en) + " ▌"

    if target_lang != "en" and full_en:
        try:
            final_translation = GoogleTranslator(source="en", target="ru").translate(full_en)
            yield final_translation
        except Exception:
            yield full_en
    else:
        yield full_en

    thread.join()
    _clear_cuda_cache()


def save_result(text: str, mode: str):
    if not text or not text.strip() or text.startswith("### 🤖"):
        return None
    ext = "txt" if mode == "full" else "md"
    fname = f"/kaggle/working/result_{mode}_{int(time.time())}.{ext}"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(text)
    return fname


def save_as_pdf(text: str) -> str:
    if not text or text.startswith("### 🤖"):
        return None
    try:
        from fpdf import FPDF
        import re

        clean_text = re.sub(r'[#*`]', '', text)

        pdf = FPDF()
        pdf.add_page()
        pdf.set_auto_page_break(auto=True, margin=15)

        # ✅ Правильный путь
        font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
        pdf.add_font("DejaVu", "", font_path)
        pdf.set_font("DejaVu", size=11)

        for line in clean_text.split('\n'):
            line = line.strip()
            if not line:
                pdf.ln(4)
                continue
            pdf.multi_cell(0, 8, line)
            pdf.ln(2)

        fname = f"/kaggle/working/analysis_{int(time.time())}.pdf"
        pdf.output(fname)
        return fname
    except Exception as e:
        print(f"PDF error: {e}")
        return None


def save_as_docx(text: str) -> str:
    """Saves analysis in DOCX with formatting."""
    if not text or text.startswith("### 🤖"):
        return None
    try:
        from docx import Document
        from docx.shared import Pt, RGBColor
        from docx.enum.text import WD_ALIGN_PARAGRAPH

        doc = Document()

        # Стиль документа
        style = doc.styles['Normal']
        style.font.name = 'Calibri'
        style.font.size = Pt(11)

        for line in text.split('\n'):
            line = line.strip()
            if not line:
                continue

            # Markdown Headings → Word Headings
            if line.startswith('### '):
                p = doc.add_heading(line[4:], level=3)
            elif line.startswith('## '):
                p = doc.add_heading(line[3:], level=2)
            elif line.startswith('# '):
                p = doc.add_heading(line[2:], level=1)
            # Списки
            elif line.startswith('- ') or line.startswith('* '):
                doc.add_paragraph(line[2:], style='List Bullet')
            elif line[0].isdigit() and line[1] in '.':
                doc.add_paragraph(line[3:], style='List Number')
            # Обычный текст
            else:
                doc.add_paragraph(line)

        fname = f"/kaggle/working/analysis_{int(time.time())}.docx"
        doc.save(fname)
        return fname
    except Exception as e:
        print(f"DOCX ошибка: {e}")
        return None


# ============================================================
# --- GRADIO UI ---
# ============================================================

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&family=Syne:wght@400;700;800&display=swap');

* { font-family: 'Syne', sans-serif !important; }
code, .mono { font-family: 'JetBrains Mono', monospace !important; }

body, .gradio-container {
    background: #0d0f14 !important;
}

.output-box {
    background: #12151d !important;
    color: #e8eaf0 !important;
    border: 1px solid #ff6600 !important;
    border-radius: 8px;
    padding: 20px;
    min-height: 420px;
    font-size: 0.95rem;
    line-height: 1.7;
}
.output-box * { color: #e8eaf0 !important; }
.output-box h1, .output-box h2, .output-box h3 { color: #ff6600 !important; }

#query-input label span { color: #ff9944 !important; font-weight: 700; }

.gr-button-primary {
    background: linear-gradient(135deg, #ff6600, #ff9900) !important;
    border: none !important;
    font-weight: 700 !important;
}
.gr-button-secondary {
    border: 1px solid #ff6600 !important;
    color: #ff6600 !important;
}


"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange")) as demo:

    gr.Markdown(
        "# 🎙️ Llama 3 · Professional Analyst Station Audio/Video to text\n"
        "*Kaggle T4 Edition — Whisper + LLaMA 3.2 3B Instruct + Pyannote"
    )
    
    with gr.Row():

        # ---- LEFT COLUMN: Input and decryption ----
        with gr.Column(scale=1):
            # with gr.Tabs():
                # with gr.TabItem("🎤 Микрофон"):
                #     mic_in = gr.Audio(
                #         sources="microphone",
                #         type="filepath",
                #         label=None,
                #     )
                # with gr.TabItem("📁 Загрузить файл"):
                #     file_in = gr.File(
                #         label="Аудио или Видео файл",
                #         file_types=[".mp3", ".wav", ".m4a", ".flac", ".ogg", ".mp4", ".avi", ".mkv", ".mov", ".webm"],
                #         type="filepath",
                #     )

            with gr.Tabs():
                with gr.TabItem("🎤 Microphone"):
                    mic_in = gr.Audio(
                        sources="microphone",
                        type="filepath",
                        label=None,
                        )
                with gr.TabItem("📁 Audio"):
                    file_in = gr.Audio(
                        sources="upload",
                        type="filepath",
                        label=None,  
                        )
                with gr.TabItem("🎬 Video"):
                    video_in = gr.Video(
                        label=None,  
                        )


            
            mode_sel = gr.Radio(
                choices=[
                    "🎙️ Standard transcript",
                    "🕐 With timecodes",
                    "👥 Identify speakers",
            ],
                value="🎙️ Standard transcript",  # by default
                label="Decryption mode",
            )

            
            txt_raw = gr.Textbox(
                label="📝 Transcript (Whisper)",
                lines=12,
                show_copy_button=True,
                placeholder="The text will appear here after decryption....",
            )

           
            with gr.Row():
                btn_fix = gr.Button("✏️ Correct grammar (LLaMA)", variant="secondary")
                btn_dl_full = gr.Button("💾 Download .txt", variant="primary")
                btn_dl_full_docx = gr.Button("📝 Download .docx", variant="primary")

            out_full = gr.File(label="Decryption file (.txt)")
            out_full_docx = gr.File(label="Decryption file (.docx)")

        
        # ---- ПРАВАЯ КОЛОНКА: анализ ----
        with gr.Column(scale=1):

            query_in = gr.Textbox(
                label="❓ Your question about the text",
                placeholder="Enter a question or leave it blank for auto-summation...",
                lines=2,
                elem_id="query-input",
            )

            # ← ДОБАВЛЕН role_sel
            role_sel = gr.Dropdown(
                choices=list(ROLES.keys()),
                value="📊 Analyst",
                label="🎭 Role",
            )

            with gr.Row():
                lang_sel = gr.Radio(
                    choices=["ru", "en"],
                    value="ru",
                    label="🌍 Language",
                    scale=1,
                )
                btn_run = gr.Button("🚀 Run analysis", variant="primary", scale=2)

            out_md = gr.Markdown(
                value="### 🤖 Waiting for request...",
                elem_classes="output-box",
            )

            with gr.Row():
                btn_dl_sum = gr.Button("💾 Download .md", variant="primary")
                btn_dl_pdf = gr.Button("📄 Download PDF", variant="primary")
                btn_dl_docx = gr.Button("📝 Download DOCX", variant="primary")
                btn_clr = gr.Button("🧹 Reset all", variant="stop")

            out_sum  = gr.File(label="Analysis (.md)")
            out_pdf  = gr.File(label="Analysis (.pdf)")
            out_docx = gr.File(label="Analysis (.docx)")

      # ---- СОБЫТИЯ ----
    mic_in.change(fn=transcribe_auto, inputs=[mic_in, mode_sel], outputs=txt_raw)
    file_in.change(fn=transcribe_auto, inputs=[file_in, mode_sel], outputs=txt_raw)
    video_in.change(fn=transcribe_auto, inputs=[video_in, mode_sel], outputs=txt_raw)
    btn_fix.click(fn=fix_grammar_llama, inputs=txt_raw, outputs=txt_raw)
    btn_dl_full.click(fn=lambda x: save_result(x, "full"), inputs=txt_raw, outputs=out_full)
    btn_dl_full_docx.click(fn=save_as_docx, inputs=txt_raw, outputs=out_full_docx)
    btn_run.click(
        fn=generate_analysis_stream,
        inputs=[txt_raw, query_in, lang_sel, role_sel],
        outputs=out_md,
    )
    btn_dl_sum.click(fn=lambda x: save_result(x, "summary"), inputs=out_md, outputs=out_sum)
    btn_dl_pdf.click(fn=save_as_pdf, inputs=out_md, outputs=out_pdf)
    btn_dl_docx.click(fn=save_as_docx, inputs=out_md, outputs=out_docx)
    btn_clr.click(
        fn=lambda: (None, None, None, "", "", "### 🤖 Expectation...", None, None, None, None, None),
        outputs=[mic_in, file_in, video_in, txt_raw, query_in, out_md, out_sum, out_full, out_full_docx, out_pdf, out_docx],
    )
# ============================================================
# --- LAUNCH ---
# ============================================================
demo.queue().launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    inline=True,
    debug=True,
)

/tmp/ipykernel_55/2404315057.py:369: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange")) as demo:
/tmp/ipykernel_55/2404315057.py:369: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="orange")) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 📊 Analyst or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://0adff2cd61b6740f3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error